# MLP do zero: backpropagation sem frameworks

Este notebook implementa uma rede neural (MLP) **inteiramente do zero**, usando apenas NumPy — sem PyTorch, sem TensorFlow, sem `autograd`.

O objetivo não é ter o melhor modelo possível, mas sim entender exatamente o que acontece dentro de uma rede neural: o forward pass, a função de perda e o backpropagation, tudo escrito manualmente.

**Dataset**: usamos o `load_digits` do scikit-learn (dígitos 8x8, 10 classes) em vez do MNIST completo (28x28), porque não depende de download externo e roda em qualquer máquina instantaneamente. A lógica é idêntica — se quiser usar o MNIST de verdade depois, basta trocar a fonte dos dados (veja a nota no final).

## 1. Imports e dados

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits()
X = digits.data / 16.0  # normaliza os pixels (0-16) para o intervalo 0-1
y = digits.target

print("Formato de X:", X.shape)   # (1797 amostras, 64 pixels cada imagem 8x8)
print("Formato de y:", y.shape)
print("Classes:", np.unique(y))


In [ ]:
# Visualizar alguns exemplos
fig, axes = plt.subplots(1, 8, figsize=(10, 2))
for i, ax in enumerate(axes):
    ax.imshow(digits.images[i], cmap="gray")
    ax.set_title(str(y[i]))
    ax.axis("off")
plt.show()


## 2. Preparar os dados

Dividimos em treino/teste e transformamos os rótulos (0-9) em **one-hot encoding** — cada rótulo vira um vetor de 10 posições com 1 na posição da classe correta e 0 nas demais. Isso é necessário porque a saída da rede é uma distribuição de probabilidade sobre as 10 classes, não um único número.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

def one_hot(y, num_classes=10):
    out = np.zeros((y.shape[0], num_classes))
    out[np.arange(y.shape[0]), y] = 1
    return out

Y_train = one_hot(y_train)
Y_test = one_hot(y_test)

print("Treino:", X_train.shape, "Teste:", X_test.shape)


## 3. Inicializar os pesos

A rede tem uma arquitetura simples: **entrada (64) → camada oculta (32, ReLU) → saída (10, softmax)**.

Os pesos são inicializados com uma variação pequena e proporcional ao tamanho da camada anterior (inicialização "He"), o que evita que os gradientes explodam ou desapareçam logo no início do treino.

In [ ]:
n_features = X_train.shape[1]  # 64
n_hidden = 32
n_classes = 10

rng = np.random.default_rng(42)
W1 = rng.normal(0, np.sqrt(2.0 / n_features), (n_features, n_hidden))
b1 = np.zeros(n_hidden)
W2 = rng.normal(0, np.sqrt(2.0 / n_hidden), (n_hidden, n_classes))
b2 = np.zeros(n_classes)

print("W1:", W1.shape, "| W2:", W2.shape)


## 4. Forward pass

Este é o caminho que os dados percorrem da entrada até a previsão:

1. `z1 = X @ W1 + b1` — combinação linear (o que cada neurônio "vê")
2. `a1 = ReLU(z1)` — não-linearidade (sem isso, empilhar camadas seria inútil — várias transformações lineares seguidas ainda são uma transformação linear)
3. `z2 = a1 @ W2 + b2` — segunda combinação linear
4. `a2 = softmax(z2)` — transforma os números em probabilidades que somam 1

Guardamos os valores intermediários (`cache`) porque vamos precisar deles no backward pass.

In [ ]:
def relu(z):
    return np.maximum(0, z)

def relu_deriv(z):
    # derivada da ReLU: 1 onde z > 0, senão 0
    return (z > 0).astype(float)

def softmax(z):
    z = z - np.max(z, axis=1, keepdims=True)  # estabilidade numérica
    exp_z = np.exp(z)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

def forward(X, W1, b1, W2, b2):
    z1 = X @ W1 + b1
    a1 = relu(z1)
    z2 = a1 @ W2 + b2
    a2 = softmax(z2)
    cache = (X, z1, a1, z2, a2)
    return a2, cache


## 5. Função de perda: cross-entropy

Mede o quão "surpresa" a rede ficou com a resposta certa. Se a rede dá alta probabilidade à classe correta, a perda é baixa; se dá baixa probabilidade, a perda é alta (cresce logaritmicamente).

In [ ]:
def cross_entropy_loss(y_pred, y_true):
    eps = 1e-9  # evita log(0)
    return -np.mean(np.sum(y_true * np.log(y_pred + eps), axis=1))


## 6. Backward pass (backpropagation manual)

Aqui está o coração do notebook. Backpropagation é a **regra da cadeia** aplicada camada por camada, de trás para frente:

- Começamos com o erro na saída: `dz2 = a2 - y_true` (uma propriedade elegante de combinar softmax + cross-entropy)
- Propagamos esse erro de volta pela segunda camada de pesos (`dW2`, `db2`)
- Continuamos propagando pela ReLU (`relu_deriv`) até a primeira camada (`dW1`, `db1`)

Cada gradiente (`dW1`, `dW2`, etc.) diz exatamente "para que lado e o quanto mexer cada peso para reduzir o erro".

In [ ]:
def backward(cache, y_true, W2):
    X, z1, a1, z2, a2 = cache
    m = X.shape[0]  # tamanho do batch

    # gradiente na saída (softmax + cross-entropy combinados)
    dz2 = (a2 - y_true) / m
    dW2 = a1.T @ dz2
    db2 = np.sum(dz2, axis=0)

    # propaga o erro de volta pela camada oculta
    da1 = dz2 @ W2.T
    dz1 = da1 * relu_deriv(z1)
    dW1 = X.T @ dz1
    db1 = np.sum(dz1, axis=0)

    return dW1, db1, dW2, db2


## 7. Loop de treino

A cada época:
1. Fazemos o forward pass e calculamos a perda
2. Calculamos os gradientes com backward pass
3. Atualizamos os pesos na direção **contrária** ao gradiente (por isso `-=`), escalados pela taxa de aprendizado (`lr`)

Isso é gradient descent na sua forma mais crua — sem otimizadores sofisticados como Adam, só o essencial.

In [ ]:
lr = 0.5
epochs = 300
losses = []

for epoch in range(epochs):
    y_pred, cache = forward(X_train, W1, b1, W2, b2)
    loss = cross_entropy_loss(y_pred, Y_train)
    losses.append(loss)

    dW1, db1, dW2, db2 = backward(cache, Y_train, W2)

    W1 -= lr * dW1
    b1 -= lr * db1
    W2 -= lr * dW2
    b2 -= lr * db2

    if epoch % 50 == 0:
        print(f"epoch {epoch:4d} | loss {loss:.4f}")


In [ ]:
plt.plot(losses)
plt.xlabel("Época")
plt.ylabel("Loss (cross-entropy)")
plt.title("Curva de treino")
plt.show()


## 8. Avaliação

In [ ]:
y_pred_test, _ = forward(X_test, W1, b1, W2, b2)
test_acc = np.mean(np.argmax(y_pred_test, axis=1) == y_test)
print(f"Acuracia no teste: {test_acc:.4f}")


In [ ]:
# Ver alguns erros de classificação, se houver
preds = np.argmax(y_pred_test, axis=1)
errors = np.where(preds != y_test)[0]
print(f"Total de erros: {len(errors)} de {len(y_test)}")

if len(errors) > 0:
    fig, axes = plt.subplots(1, min(5, len(errors)), figsize=(10, 2))
    if len(errors) == 1:
        axes = [axes]
    for ax, idx in zip(axes, errors[:5]):
        img = X_test[idx].reshape(8, 8)
        ax.imshow(img, cmap="gray")
        ax.set_title(f"real:{y_test[idx]} prev:{preds[idx]}")
        ax.axis("off")
    plt.show()


## 9. Bônus: a mesma rede em PyTorch

Depois de implementar tudo manualmente, vale a pena ver o quanto um framework abstrai. O código abaixo faz exatamente a mesma coisa que todo o notebook acima, em poucas linhas.

> Requer `pip install torch` — não incluído por padrão neste ambiente. Rode localmente para comparar.

In [ ]:
import torch
import torch.nn as nn

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

model = nn.Sequential(
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 10),
)

optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(300):
    optimizer.zero_grad()
    logits = model(X_train_t)
    loss = loss_fn(logits, y_train_t)
    loss.backward()       # <- backpropagation automatico (o que fizemos a mao acima)
    optimizer.step()
    if epoch % 50 == 0:
        print(f"epoch {epoch:4d} | loss {loss.item():.4f}")

with torch.no_grad():
    preds = model(X_test_t).argmax(dim=1)
    acc = (preds == y_test_t).float().mean()
    print(f"\nTest accuracy: {acc:.4f}")


## Notas finais

- **Para usar o MNIST completo (28x28)** em vez do dataset simplificado: substitua o carregamento de dados por
  ```python
  from sklearn.datasets import fetch_openml
  mnist = fetch_openml("mnist_784", version=1)
  ```
  (requer conexão à internet e é mais lento para baixar na primeira vez).

- **Experimentos sugeridos** para aprofundar o entendimento:
  - Mude `n_hidden` (32 → 8 ou → 128) e observe o efeito na acurácia e na velocidade de treino
  - Mude a taxa de aprendizado (`lr`) para um valor muito alto (ex: 5.0) ou muito baixo (ex: 0.001) e observe o que acontece com a curva de perda
  - Adicione uma segunda camada oculta e refaça o backward pass manualmente — é o melhor exercício para realmente fixar a regra da cadeia
  - Implemente mini-batches em vez de usar o dataset de treino inteiro a cada época
